<a href="https://colab.research.google.com/github/Haripriya-vh-code/Passenger_seat_prediction/blob/main/redbus_hpjan_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib as plt
import numpy as np
from sklearn.model_selection import train_test_split
df1 = pd.read_csv("/train.csv")
df2= pd.read_csv("/transactions.csv",on_bad_lines='skip')
df2.head()

,doj,doi,srcid,destid,srcid_region,destid_region,srcid_tier,destid_tier,cumsum_seatcount,cumsum_searchcount,dbd
0,2023-03-01,2023-01-30,45,46,Karnataka,Tamil Nadu,Tier 1,Tier 1,8.0,76.0,30.0
1,2023-03-01,2023-01-30,46,45,Tamil Nadu,Karnataka,Tier 1,Tier 1,8.0,70.0,30.0
2,2023-03-01,2023-01-30,45,47,Karnataka,Andhra Pradesh,Tier 1,Tier 1,4.0,142.0,30.0
3,2023-03-01,2023-01-30,47,45,Andhra Pradesh,Karnataka,Tier 1,Tier 1,0.0,68.0,30.0
4,2023-03-01,2023-01-30,46,9,Tamil Nadu,Tamil Nadu,Tier 1,Tier2,9.0,162.0,30.0


In [ ]:
df2['doj'] = pd.to_datetime(df2['doj'], errors='coerce')
df2['doi'] = pd.to_datetime(df2['doi'], errors='coerce')
df2['dbd'] = (df2['doj'] - df2['doi']).dt.days

In [ ]:
# Step 1: Create month mapping for Dutch to English
month_map = {
    'jan': 'Jan', 'feb': 'Feb', 'mrt': 'Mar', 'apr': 'Apr',
    'mei': 'May', 'jun': 'Jun', 'jul': 'Jul', 'aug': 'Aug',
    'sep': 'Sep', 'okt': 'Oct', 'nov': 'Nov', 'dec': 'Dec'
}

# Step 2: Function to replace Dutch months with English
def convert_dutch_months(date_str):
    if pd.isna(date_str):
        return date_str
    date_str = date_str.lower()
    for dutch, eng in month_map.items():
        if dutch in date_str:
            return date_str.replace(dutch, eng)
    return date_str

In [ ]:
df2['doi'] = df2['doi'].astype(str)
df2['doi_cleaned'] = df2['doi'].apply(convert_dutch_months)
df2['doi_cleaned'] = pd.to_datetime(df2['doi_cleaned'], dayfirst=True, errors='coerce')
df2['doj'] = pd.to_datetime(df2['doj'], errors='coerce')  # if not already converted
df2['dbd'] = (df2['doj'] - df2['doi_cleaned']).dt.days


/tmp/ipython-input-5-1242620794.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df2['doi_cleaned'] = pd.to_datetime(df2['doi_cleaned'], dayfirst=True, errors='coerce')


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests


# Step 1: Define Dutch to English month mapping
month_map = {
    'jan': 'Jan', 'feb': 'Feb', 'mrt': 'Mar', 'apr': 'Apr',
    'mei': 'May', 'jun': 'Jun', 'jul': 'Jul', 'aug': 'Aug',
    'sep': 'Sep', 'okt': 'Oct', 'nov': 'Nov', 'dec': 'Dec'
}

# Step 2: Function to replace Dutch months
def convert_dutch_months(date_str):
    date_str = date_str.lower()
    for dutch, eng in month_map.items():
        if dutch in date_str:
            return date_str.replace(dutch, eng)
    return date_str

# Function to extract holiday data
def extract_holidays(url, year_label):
    response = requests.get(url)
    soup = BeautifulSoup(response.content, "html.parser")
    table = soup.find("table", id="holidays-table")
    rows = table.find_all("tr")

    data = []
    for row in rows[1:]:
        cols = row.find_all(["th", "td"])
        if len(cols) == 4:
            raw_date = cols[0].get_text(strip=True) + f" {year_label}"
            cleaned_date = convert_dutch_months(raw_date)
            name = cols[2].get_text(strip=True)
            type_ = cols[3].get_text(strip=True)

            if type_ in ["Gazetted Holiday", "Restricted Holiday"]:
                date_obj = pd.to_datetime(cleaned_date, dayfirst=True, errors='coerce')
                data.append({
                    f"Date_{year_label}": date_obj,
                    "Holiday Name": name,
                    "Type": type_
                })
    return pd.DataFrame(data)

# Step 3: Load all 3 years
df2025 = extract_holidays("https://www.timeanddate.com/holidays/india/2025", "2025")
df2024 = extract_holidays("https://www.timeanddate.com/holidays/india/2024", "2024")
df2023 = extract_holidays("https://www.timeanddate.com/holidays/india/2023", "2023")


In [ ]:
#Merging datasets

df_train= pd.merge(df2024,df2023,on="Holiday Name",how='outer')
df_train =pd.merge(df_train,df2025,on="Holiday Name",how='outer')
df_train.head()
df_train= df_train.drop(['Type_x','Type_y',],axis=1)
df_train.head()

,Date_2024,Holiday Name,Date_2023,Date_2025,Type
0,NaT,Bahag Bihu/Vaisakhadi,NaT,2025-04-15,Restricted Holiday
1,2024-06-17,Bakrid,2023-06-29,2025-06-07,Gazetted Holiday
2,2024-11-03,Bhai Duj,2023-11-15,2025-10-23,Restricted Holiday
3,2024-05-08,Birthday of Rabindranath,2023-05-09,2025-05-09,Restricted Holiday
4,2024-05-23,Buddha Purnima/Vesak,2023-05-05,2025-05-12,Gazetted Holiday


In [ ]:
df_train['Date_2024'] = df_train['Date_2024'].fillna(df_train['Date_2023'])
df_train['Date_2024'] = df_train['Date_2024'].fillna(df_train['Date_2025'])

df_train['Date_2023'] = df_train['Date_2023'].fillna(df_train['Date_2024'])
df_train['Date_2023'] = df_train['Date_2023'].fillna(df_train['Date_2025'])

df_train['Date_2025']   = df_train['Date_2025'].fillna(df_train['Date_2023'])
df_train['Date_2025']   = df_train['Date_2025'].fillna(df_train['Date_2024'])
df_train.loc[[29, 40], 'Type'] = 'Restricted Holiday'
df_train['Type'] = df_train['Type'].fillna('Gazetted Holiday')
df_train.head(7)

,Date_2024,Holiday Name,Date_2023,Date_2025,Type
0,2025-04-15,Bahag Bihu/Vaisakhadi,2025-04-15,2025-04-15,Restricted Holiday
1,2024-06-17,Bakrid,2023-06-29,2025-06-07,Gazetted Holiday
2,2024-11-03,Bhai Duj,2023-11-15,2025-10-23,Restricted Holiday
3,2024-05-08,Birthday of Rabindranath,2023-05-09,2025-05-09,Restricted Holiday
4,2024-05-23,Buddha Purnima/Vesak,2023-05-05,2025-05-12,Gazetted Holiday
5,2024-04-09,Chaitra Sukhladi,2023-03-22,2025-03-30,Restricted Holiday
6,2024-11-07,Chhat Puja (Pratihar Sashthi/Surya Sashthi),2023-11-19,2025-10-28,Restricted Holiday


In [ ]:
df2['route']= df1['srcid'].astype(str)+'-'+ df1['destid'].astype(str)
df2.head()

,doj,doi,srcid,destid,srcid_region,destid_region,srcid_tier,destid_tier,cumsum_seatcount,cumsum_searchcount,dbd,doi_cleaned,route
0,2023-03-01,2023-01-30,45,46,Karnataka,Tamil Nadu,Tier 1,Tier 1,8.0,76.0,30,2023-01-30,45-46
1,2023-03-01,2023-01-30,46,45,Tamil Nadu,Karnataka,Tier 1,Tier 1,8.0,70.0,30,2023-01-30,46-45
2,2023-03-01,2023-01-30,45,47,Karnataka,Andhra Pradesh,Tier 1,Tier 1,4.0,142.0,30,2023-01-30,45-47
3,2023-03-01,2023-01-30,47,45,Andhra Pradesh,Karnataka,Tier 1,Tier 1,0.0,68.0,30,2023-01-30,47-45
4,2023-03-01,2023-01-30,46,9,Tamil Nadu,Tamil Nadu,Tier 1,Tier2,9.0,162.0,30,2023-01-30,46-9


In [ ]:
# only keep rows with valid dates

df2['doj'] = pd.to_datetime(df2['doj'], errors='coerce')
invalid_dates = df2[df2['doj'].isna()]
df2 = df2.dropna(subset=['doj'])
df1['doj'] = pd.to_datetime(df1['doj'])
df2['doj'] = pd.to_datetime(df2['doj'])
df = pd.merge(df1, df2, on=['doj', 'srcid', 'destid'], how='left')

In [ ]:
all_holidays = pd.concat([
    df_train[['Date_2023', 'Type']].rename(columns={'Date_2023': 'date'}),
    df_train[['Date_2024', 'Type']].rename(columns={'Date_2024': 'date'}),
    df_train[['Date_2025', 'Type']].rename(columns={'Date_2025': 'date'}),
], ignore_index=True)

# Drop any nulls
all_holidays = all_holidays.dropna(subset=['date'])

# Drop duplicates
all_holidays = all_holidays.drop_duplicates(subset=['date'])

# Mark as holiday
all_holidays['is_holiday'] = 1


In [ ]:
df = df.merge(all_holidays[['date', 'is_holiday']], left_on='doj', right_on='date', how='left')
df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)
df.drop(columns=['date'], inplace=True)  # optional cleanup
df['dayofweek'] = df['doj'].dt.dayofweek         # 0 = Monday
df['month'] = df['doj'].dt.month
df['is_weekend'] = df['dayofweek'].isin([5, 6]).astype(int)
df['route'] = df['srcid'].astype(str) + '-' + df['destid'].astype(str)

feature_cols = ['srcid', 'destid', 'cumsum_searchcount', 'cumsum_seatcount',
                'is_holiday', 'dayofweek', 'month', 'is_weekend']

X = df[feature_cols]
y = df['final_seatcount']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        print(col)
        print(X_train[col].unique()[:5])  # print first few unique values

In [ ]:
# Show all non-numeric columns in X_train
non_numeric_cols = X_train.select_dtypes(include='object').columns
print("Non-numeric columns:", non_numeric_cols)


Non-numeric columns: Index([], dtype='object')


In [ ]:
for col in non_numeric_cols:
    print(f"\nColumn: {col}")
    print(X_train[col].unique()[:10])

In [ ]:
for col in non_numeric_cols:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype(str)  # ensure it's string
        X_train[col] = X_train[col].str.replace('..', '.', regex=False)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')

In [ ]:
for col in non_numeric_cols:
    X_train[col] = X_train[col].astype('category').cat.codes
X_train = X_train.fillna(0)

In [ ]:
print(df1.columns)

Index(['doj', 'srcid', 'destid', 'final_seatcount'], dtype='object')


In [ ]:
import pandas as pd

# Ensure doj is in datetime format
df1['doj'] = pd.to_datetime(df1['doj'], errors='coerce')

# Create new features
df1['dayofweek'] = df1['doj'].dt.dayofweek
df1['month'] = df1['doj'].dt.month
df1['is_weekend'] = df1['dayofweek'].isin([5, 6]).astype(int)

# Create 'dbd' (days before departure), assuming you want a fixed reference date
df1['dbd'] = (df1['doj'] - pd.to_datetime('today')).dt.days

# Create and encode the route
df1['route'] = df1['srcid'].astype(str) + '-' + df1['destid'].astype(str)
df1['route'] = df1['route'].astype('category').cat.codes

In [ ]:
# Convert to string and fix bad formats like '112..0'
X_train['cumsum_seatcount'] = X_train['cumsum_seatcount'].astype(str).str.replace('..', '.', regex=False)

# Convert to numeric, forcing errors to NaN
X_train['cumsum_seatcount'] = pd.to_numeric(X_train['cumsum_seatcount'], errors='coerce')

# Fill NaNs with 0
X_train['cumsum_seatcount'] = X_train['cumsum_seatcount'].fillna(0)

In [ ]:
X_train['srcid'] = X_train['srcid'].astype('category').cat.codes
X_train['destid'] = X_train['destid'].astype('category').cat.codes

In [ ]:
# Fix double dots like '112..0' in ALL object columns
for col in X_train.columns:
    if X_train[col].dtype == 'object':
        X_train[col] = X_train[col].astype(str).str.replace('..', '.', regex=False)
        X_train[col] = pd.to_numeric(X_train[col], errors='coerce')  # convert to float

# Handle any leftover NaNs
X_train = X_train.fillna(0)


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error




In [ ]:
!pip install --upgrade xgboost

In [ ]:
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# Fit without early stopping
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011190 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 624
[LightGBM] [Info] Number of data points in the train set: 93529, number of used features: 8
[LightGBM] [Info] Start training from score 1846.883213


LGBMRegressor(colsample_bytree=0.8, learning_rate=0.05, max_depth=8,
              n_estimators=1000, n_jobs=-1, random_state=42, subsample=0.8)

In [ ]:
print(X_train[X_train.applymap(lambda x: isinstance(x, str))].dropna(how='all'))

Empty DataFrame
Columns: [srcid, destid, cumsum_searchcount, cumsum_seatcount, is_holiday, dayofweek, month, is_weekend]
Index: []


/tmp/ipython-input-28-3472969641.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  print(X_train[X_train.applymap(lambda x: isinstance(x, str))].dropna(how='all'))


In [ ]:
y_train = pd.to_numeric(y_train, errors='coerce').fillna(0)

In [ ]:
print(df_train.columns)

Index(['Date_2024', 'Holiday Name', 'Date_2023', 'Date_2025', 'Type'], dtype='object')


In [ ]:
import pandas as pd

# Load datasets
df1 = pd.read_csv("/train.csv")          # Main training data
df2 = pd.read_csv("/transactions.csv", on_bad_lines='skip')  # Extra features

# Ensure correct datetime format
df1['doj'] = pd.to_datetime(df1['doj'], errors='coerce')
df2['doj'] = pd.to_datetime(df2['doj'], errors='coerce')

# Create route key in df2 (optional but useful)
df2['route'] = df2['srcid'].astype(str) + '-' + df2['destid'].astype(str)

# Merge extra features from df2 into df1
merge_cols = ['srcid', 'destid', 'doj', 'cumsum_seatcount', 'cumsum_searchcount']
df_train = df1.merge(df2[merge_cols].drop_duplicates(), on=['srcid', 'destid', 'doj'], how='left')

# Drop rows with missing target or features
df_train = df_train.dropna(subset=['final_seatcount', 'cumsum_seatcount', 'cumsum_searchcount'])

# Feature Engineering
df_train['dayofweek'] = df_train['doj'].dt.dayofweek
df_train['month'] = df_train['doj'].dt.month
df_train['is_weekend'] = df_train['dayofweek'].isin([5, 6]).astype(int)

# Final features for training
feature_cols = ['srcid', 'destid', 'cumsum_searchcount', 'cumsum_seatcount',
                'dayofweek', 'month', 'is_weekend']
X = df_train[feature_cols]
y = df_train['final_seatcount']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from lightgbm import LGBMRegressor
import numpy as np

# Log transform target to reduce skew
y_log = np.log1p(y)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)

# Train LightGBM model
model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predict
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

# Evaluate
print("MAE:", mean_absolute_error(y_test_actual, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test_actual, y_pred)))



[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.020890 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 617
[LightGBM] [Info] Number of data points in the train set: 264176, number of used features: 7
[LightGBM] [Info] Start training from score 7.429006
MAE: 268.5383840601987
RMSE: 394.1030051143711


In [ ]:
df_test = pd.read_csv('/test_1.csv')

In [ ]:
df_test_raw = pd.read_csv("/test_1.csv")
print("Raw test shape:", df_test_raw.shape)


Raw test shape: (5900, 4)


In [ ]:
print("Duplicate rows:", df_test.duplicated().sum())
df_test = df_test.drop_duplicates()
print("After removing duplicates:", df_test.shape)
df_test['doj'] = pd.to_datetime(df_test['doj'], errors='coerce', dayfirst=True)

Duplicate rows: 0
After removing duplicates: (5900, 4)


In [ ]:
# Convert to datetime
df_train['doj'] = pd.to_datetime(df_train['doj'], errors='coerce')
df_test['doj'] = pd.to_datetime(df_test['doj'], errors='coerce')
df_test = df_test.dropna(subset=['doj'])  # Drop rows where doj is NaT

# Add date features to train
df_train['dayofweek'] = df_train['doj'].dt.dayofweek
df_train['month'] = df_train['doj'].dt.month
df_train['is_weekend'] = df_train['dayofweek'].isin([5, 6]).astype(int)

# Add same features to test
df_test['dayofweek'] = df_test['doj'].dt.dayofweek
df_test['month'] = df_test['doj'].dt.month
df_test['is_weekend'] = df_test['dayofweek'].isin([5, 6]).astype(int)

In [ ]:
import pandas as pd

# Load the transactions file
df_trans = pd.read_csv('/transactions.csv', on_bad_lines='skip')

# Convert doj to datetime
df_trans['doj'] = pd.to_datetime(df_trans['doj'], errors='coerce')

# Define columns needed for merging
merge_cols = ['srcid', 'destid', 'doj', 'srcid_region', 'destid_region',
              'srcid_tier', 'destid_tier', 'cumsum_seatcount', 'cumsum_searchcount']

# Drop duplicates to avoid merge issues
df_trans = df_trans.drop_duplicates(subset=merge_cols)

# Ensure df_test is already loaded and has matching columns
df_test = df_test.merge(df_trans[merge_cols], on=['srcid', 'destid', 'doj'], how='left')

In [ ]:

df_trans['doj'] = pd.to_datetime(df_trans['doj'], errors='coerce')
merge_cols = ['srcid', 'destid', 'doj', 'srcid_region', 'destid_region',
              'srcid_tier', 'destid_tier', 'cumsum_seatcount', 'cumsum_searchcount']

df_trans = df_trans.drop_duplicates(subset=merge_cols)
df_test = df_test.merge(df_trans[merge_cols], on=['srcid', 'destid', 'doj'], how='left')

In [ ]:
df_test.fillna(0, inplace=True)
# Add route to df_train to enable merge
df_train['doj'] = pd.to_datetime(df_train['doj'], errors='coerce')
df_trans['doj'] = pd.to_datetime(df_trans['doj'], errors='coerce')

merge_cols = ['srcid', 'destid', 'doj', 'cumsum_seatcount', 'cumsum_searchcount']
df_trans_unique = df_trans[merge_cols].drop_duplicates()

# Merge into train
df_train = df_train.merge(df_trans_unique, on=['srcid', 'destid', 'doj'], how='left')
df_train.fillna(0, inplace=True)

# Recreate features
df_train['dayofweek'] = df_train['doj'].dt.dayofweek
df_train['month'] = df_train['doj'].dt.month
df_train['is_weekend'] = df_train['dayofweek'].isin([5, 6]).astype(int)

/tmp/ipython-input-43-3352874725.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_test.fillna(0, inplace=True)


In [ ]:
print(df_train.columns.tolist())

['doj', 'srcid', 'destid', 'final_seatcount', 'cumsum_seatcount', 'cumsum_searchcount', 'dayofweek', 'month', 'is_weekend']


In [ ]:
# Drop existing conflicting columns from df_test if present
df_test = df_test.drop(columns=['cumsum_seatcount', 'cumsum_searchcount'], errors='ignore')

# Now perform the merge safely
df_test = df_test.merge(
    df_trans[merge_cols].drop_duplicates(),
    on=['srcid', 'destid', 'doj'],
    how='left'
)

In [ ]:
df_test['dbd'] = (df_test['doj'] - pd.to_datetime('today')).dt.days
df_test['dayofweek'] = df_test['doj'].dt.dayofweek
df_test['month'] = df_test['doj'].dt.month
df_test['is_weekend'] = df_test['dayofweek'].isin([5, 6]).astype(int)
df_test['route'] = df_test['srcid'].astype(str) + '-' + df_test['destid'].astype(str)
df_test['route'] = df_test['route'].astype('category').cat.codes

In [ ]:
# Use same features as used in training
feature_cols = ['srcid', 'destid', 'cumsum_searchcount', 'cumsum_seatcount',
                'dayofweek', 'month', 'is_weekend']  # Only 7 features!

df_test['doj'] = pd.to_datetime(df_test['doj'], errors='coerce')
df_test = df_test.dropna(subset=['doj'])

df_test['dbd'] = (df_test['doj'] - pd.to_datetime('today')).dt.days
df_test['dayofweek'] = df_test['doj'].dt.dayofweek
df_test['month'] = df_test['doj'].dt.month
df_test['is_weekend'] = df_test['dayofweek'].isin([5, 6]).astype(int)
df_test['route'] = df_test['srcid'].astype(str) + '-' + df_test['destid'].astype(str)
df_test['route'] = df_test['route'].astype('category').cat.codes
X_test_final = df_test[feature_cols]
test_pred_log = model.predict(X_test_final)
test_pred = np.expm1(test_pred_log)  # if model was trained on log1p(y)

In [ ]:
df_test = df_test.drop_duplicates()
print("After removing duplicates:", df_test.shape)

After removing duplicates: (5900, 23)


In [ ]:
submission = pd.DataFrame({
    'route_key': df_test['route_key'].values,  # 2400 entries
    'final_seatcount': test_pred[:len(df_test)]  # truncate to match
})
submission.to_csv("submission_file.csv", index=False)

In [ ]:
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>